In [2]:
import torch
from pathlib import Path

## loading data

In [3]:
tokens_dir = Path("../model-comparison/my_results/tokens")
token_pt_files = sorted(tokens_dir.glob("*.pt"))

tokens = {}
for f in token_pt_files:
    tokens[f.stem] = torch.load(f)

In [ ]:
import torch
import numpy as np
import pandas as pd
from scipy.stats import levene, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from pathlib import Path

BASE_DIR = Path("../model-comparison/my_results/tokens")
SHUFFLED_SUFFIX = "_shuffled"
ALPHA = 0.05

# --- Load all data ---
pt_files = sorted(BASE_DIR.glob("*.pt"))
data = {f.stem: torch.load(f) for f in pt_files}

# --- Auto-pair originals with their shuffled versions ---
pairs = []
for name in data.keys():
    if name.endswith(SHUFFLED_SUFFIX):
        base = name[: -len(SHUFFLED_SUFFIX)]
        if base in data:
            pairs.append((base, name))
pairs = sorted(set(pairs))

def to_float32(t):
    return t.detach().to(dtype=torch.float32, device="cpu").numpy()

def stack_batches(list_of_tensors):
    """Stack list of 1D tensors -> shape (n_batches, n_layers)"""
    return np.stack([to_float32(t).ravel() for t in list_of_tensors])

for orig_name, shuf_name in pairs:
    A = stack_batches(data[orig_name])     # shape (100, 35)
    B = stack_batches(data[shuf_name])     # shape (100, 35)
    
    n_batches, n_layers = A.shape

    layer_ix = []
    var_A = []
    var_B = []
    pvals = []

    # Test variance per LAYER (across 100 batches)
    for i in range(n_layers):
        a = A[:, i]
        b = B[:, i]
        vA = np.var(a, ddof=0)
        vB = np.var(b, ddof=0)
        _, p = levene(a, b, center='median')  # Brown–Forsythe
        layer_ix.append(i)
        var_A.append(vA)
        var_B.append(vB)
        pvals.append(p)

    rej, p_adj, _, _ = multipletests(pvals, alpha=ALPHA, method='fdr_bh')

    df = pd.DataFrame({
        "layer": layer_ix,
        "var_original": var_A,
        "var_shuffled": var_B,
        "levene_p": pvals,
        "levene_p_fdr": p_adj,
        "different_variance_FDR": rej
    })

    # Global summary across layers
    u_stat, p_less = mannwhitneyu(var_A, var_B, alternative="less")

    print(f"\n=== {orig_name} vs {shuf_name} ===")
    print(f"{n_layers} layers, {n_batches} batches per layer")
    print(f"Mean variance original: {np.mean(var_A):.6f}")
    print(f"Mean variance shuffled: {np.mean(var_B):.6f}")
    print(f"Mann–Whitney p (orig < shuf): {p_less:.3e}")
    print(f"Layers with FDR<0.05: {np.sum(rej)}/{n_layers}")

    #df.to_csv(BASE_DIR / f"variance_per_layer_{orig_name}__vs__{shuf_name}.csv", index=False)



=== gpt4_para1 vs gpt4_para1_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000057
Mean variance shuffled: 0.000160
Mann–Whitney p (orig < shuf): 6.872e-12
Layers with FDR<0.05: 27/32

=== gpt4_para2 vs gpt4_para2_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000046
Mean variance shuffled: 0.000153
Mann–Whitney p (orig < shuf): 3.256e-12
Layers with FDR<0.05: 29/32

=== gpt4_para3 vs gpt4_para3_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000029
Mean variance shuffled: 0.000145
Mann–Whitney p (orig < shuf): 3.250e-12
Layers with FDR<0.05: 32/32

=== gpt5_para1 vs gpt5_para1_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000061
Mean variance shuffled: 0.000146
Mann–Whitney p (orig < shuf): 2.959e-11
Layers with FDR<0.05: 27/32

=== gpt5_para2 vs gpt5_para2_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000062
Mean variance shuffled: 0.000155
Mann–Whitney p (or

In [21]:
import torch
import numpy as np
import pandas as pd
from scipy.stats import levene, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from pathlib import Path

# ------------------------ Config ------------------------
BASE_DIR = Path("../model-comparison/my_results/tokens")
SHUFFLED_SUFFIX = "_shuffled"
ALPHA = 0.05

# -------------------- Utilities -------------------------
def to_float32(t):
    return t.detach().to(dtype=torch.float32, device="cpu").numpy()

def stack_batches(list_of_tensors):
    """Stack list of 1D tensors -> array shape (n_batches, n_layers)."""
    return np.stack([to_float32(t).ravel() for t in list_of_tensors])

def cliffs_delta(x: np.ndarray, y: np.ndarray):
    """
    Cliff's delta (nonparametric effect size).
    Returns (delta, magnitude) where magnitude is one of:
    'negligible','small','medium','large' (Romano et al., 2006).
    """
    x = np.asarray(x)
    y = np.asarray(y)
    # O(n*m) is fine for ~100x100
    gt = 0
    lt = 0
    for xi in x:
        gt += np.sum(xi > y)
        lt += np.sum(xi < y)
    n1 = x.size
    n2 = y.size
    delta = (gt - lt) / (n1 * n2)
    a = abs(delta)
    if a < 0.147:
        m = "negligible"
    elif a < 0.33:
        m = "small"
    elif a < 0.474:
        m = "medium"
    else:
        m = "large"
    return float(delta), m

def rank_biserial_from_u(u: float, n1: int, n2: int, orientation="x_less_y"):
    """
    Rank-biserial correlation from Mann–Whitney U.
    orientation='x_less_y' means positive values imply x < y (typical for 'less').
    """
    r = 1 - (2.0 * u) / (n1 * n2)
    return float(r) if orientation == "x_less_y" else float(-r)

# ------------------ Load and pair data -------------------
pt_files = sorted(BASE_DIR.glob("*.pt"))
data = {f.stem: torch.load(f) for f in pt_files}

pairs = []
for name in data.keys():
    if name.endswith(SHUFFLED_SUFFIX):
        base = name[: -len(SHUFFLED_SUFFIX)]
        if base in data:
            pairs.append((base, name))
pairs = sorted(set(pairs))

# ---------------------- Analysis -------------------------
for orig_name, shuf_name in pairs:
    A = stack_batches(data[orig_name])     # shape (n_batches, n_layers)
    B = stack_batches(data[shuf_name])     # shape (n_batches, n_layers)

    n_batches, n_layers = A.shape

    rows = []
    pvals = []

    # Per-layer tests/effect sizes (across batches)
    for i in range(n_layers):
        a = A[:, i]
        b = B[:, i]

        # Variances
        vA = np.var(a, ddof=0)
        vB = np.var(b, ddof=0)

        # Brown–Forsythe (Levene centered at median)
        _, p_bf = levene(a, b, center="median")
        pvals.append(p_bf)

        # Mann–Whitney (a vs. b) both one-sided and two-sided if you want
        u_stat, p_less = mannwhitneyu(a, b, alternative="less")  # orig < shuf

        # Rank–biserial correlation from U
        r_rb = rank_biserial_from_u(u_stat, len(a), len(b), orientation="x_less_y")

        # Cliff’s delta
        delta, delta_mag = cliffs_delta(a, b)

        rows.append({
            "layer": i,
            "var_original": vA,
            "var_shuffled": vB,
            "bf_p": p_bf,                          # Brown–Forsythe p-value (per layer)
            "mw_u": u_stat,
            "mw_p_less": p_less,                   # P(orig < shuf)
            "rank_biserial": r_rb,                 # [-1,1], sign consistent with 'less'
            "cliffs_delta": delta,                 # [-1,1]
            "cliffs_magnitude": delta_mag
        })

    # FDR across layers for BF test
    rej, p_adj, _, _ = multipletests(pvals, alpha=ALPHA, method="fdr_bh")
    for r, p_corr, flag in zip(rows, p_adj, rej):
        r["bf_p_fdr"] = p_corr
        r["different_variance_FDR"] = bool(flag)

    df = pd.DataFrame(rows)

    # -------- Global summary across layers (variances arrays) --------
    var_A = df["var_original"].to_numpy()
    var_B = df["var_shuffled"].to_numpy()

    u_stat_global, p_less_global = mannwhitneyu(var_A, var_B, alternative="less")
    r_rb_global = rank_biserial_from_u(u_stat_global, len(var_A), len(var_B), orientation="x_less_y")
    delta_global, delta_mag_global = cliffs_delta(var_A, var_B)

    print(f"\n=== {orig_name} vs {shuf_name} ===")
    print(f"{n_layers} layers, {n_batches} batches per layer")
    print(f"Mean variance original: {np.mean(var_A):.8f}")
    print(f"Mean variance shuffled: {np.mean(var_B):.8f}")
    print(f"Mann–Whitney p (orig < shuf): {p_less_global:.3e}")
    print(f"Rank–biserial (variances): {r_rb_global:.3f}")
    print(f"Cliff's delta (variances): {delta_global:.3f} [{delta_mag_global}]")
    print(f"Layers with BF FDR<0.05: {df['different_variance_FDR'].sum()}/{n_layers}")



=== gpt4_para1 vs gpt4_para1_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.00005677
Mean variance shuffled: 0.00015954
Mann–Whitney p (orig < shuf): 6.872e-12
Rank–biserial (variances): 0.984
Cliff's delta (variances): -0.984 [large]
Layers with BF FDR<0.05: 27/32

=== gpt4_para2 vs gpt4_para2_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.00004550
Mean variance shuffled: 0.00015313
Mann–Whitney p (orig < shuf): 3.256e-12
Rank–biserial (variances): 1.000
Cliff's delta (variances): -1.000 [large]
Layers with BF FDR<0.05: 29/32

=== gpt4_para3 vs gpt4_para3_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.00002882
Mean variance shuffled: 0.00014458
Mann–Whitney p (orig < shuf): 3.250e-12
Rank–biserial (variances): 1.000
Cliff's delta (variances): -1.000 [large]
Layers with BF FDR<0.05: 32/32

=== gpt5_para1 vs gpt5_para1_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.00006133
Mean variance

In [16]:
BASE_DIR = Path("../model-comparison/my_results/words")
SHUFFLED_SUFFIX = "_shuffled"
ALPHA = 0.05

# --- Load all data ---
pt_files = sorted(BASE_DIR.glob("*.pt"))
data = {f.stem: torch.load(f) for f in pt_files}

# --- Auto-pair originals with their shuffled versions ---
pairs = []
for name in data.keys():
    if name.endswith(SHUFFLED_SUFFIX):
        base = name[: -len(SHUFFLED_SUFFIX)]
        if base in data:
            pairs.append((base, name))
pairs = sorted(set(pairs))

def to_float32(t):
    return t.detach().to(dtype=torch.float32, device="cpu").numpy()

def stack_batches(list_of_tensors):
    """Stack list of 1D tensors -> shape (n_batches, n_layers)"""
    return np.stack([to_float32(t).ravel() for t in list_of_tensors])

for orig_name, shuf_name in pairs:
    A = stack_batches(data[orig_name])     # shape (100, 35)
    B = stack_batches(data[shuf_name])     # shape (100, 35)
    
    n_batches, n_layers = A.shape

    layer_ix = []
    var_A = []
    var_B = []
    pvals = []

    # Test variance per LAYER (across 100 batches)
    for i in range(n_layers):
        a = A[:, i]
        b = B[:, i]
        vA = np.var(a, ddof=0)
        vB = np.var(b, ddof=0)
        _, p = levene(a, b, center='median')  # Brown–Forsythe
        layer_ix.append(i)
        var_A.append(vA)
        var_B.append(vB)
        pvals.append(p)

    rej, p_adj, _, _ = multipletests(pvals, alpha=ALPHA, method='fdr_bh')

    df = pd.DataFrame({
        "layer": layer_ix,
        "var_original": var_A,
        "var_shuffled": var_B,
        "levene_p": pvals,
        "levene_p_fdr": p_adj,
        "different_variance_FDR": rej
    })

    # Global summary across layers
    u_stat, p_less = mannwhitneyu(var_A, var_B, alternative="less")

    print(f"\n=== {orig_name} vs {shuf_name} ===")
    print(f"{n_layers} layers, {n_batches} batches per layer")
    print(f"Mean variance original: {np.mean(var_A):.6f}")
    print(f"Mean variance shuffled: {np.mean(var_B):.6f}")
    print(f"Mann–Whitney p (orig < shuf): {p_less:.3e}")
    print(f"Layers with FDR<0.05: {np.sum(rej)}/{n_layers}")


=== gpt4_para1 vs gpt4_para1_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000057
Mean variance shuffled: 0.000153
Mann–Whitney p (orig < shuf): 1.800e-11
Layers with FDR<0.05: 28/32

=== gpt4_para2 vs gpt4_para2_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000046
Mean variance shuffled: 0.000151
Mann–Whitney p (orig < shuf): 3.252e-12
Layers with FDR<0.05: 29/32

=== gpt4_para3 vs gpt4_para3_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000029
Mean variance shuffled: 0.000118
Mann–Whitney p (orig < shuf): 3.252e-12
Layers with FDR<0.05: 29/32

=== gpt5_para1 vs gpt5_para1_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000061
Mean variance shuffled: 0.000168
Mann–Whitney p (orig < shuf): 1.883e-11
Layers with FDR<0.05: 29/32

=== gpt5_para2 vs gpt5_para2_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000062
Mean variance shuffled: 0.000152
Mann–Whitney p (or

In [17]:
BASE_DIR = Path("../model-comparison/my_results/sentences")
SHUFFLED_SUFFIX = "_shuffled"
ALPHA = 0.05

# --- Load all data ---
pt_files = sorted(BASE_DIR.glob("*.pt"))
data = {f.stem: torch.load(f) for f in pt_files}

# --- Auto-pair originals with their shuffled versions ---
pairs = []
for name in data.keys():
    if name.endswith(SHUFFLED_SUFFIX):
        base = name[: -len(SHUFFLED_SUFFIX)]
        if base in data:
            pairs.append((base, name))
pairs = sorted(set(pairs))

def to_float32(t):
    return t.detach().to(dtype=torch.float32, device="cpu").numpy()

def stack_batches(list_of_tensors):
    """Stack list of 1D tensors -> shape (n_batches, n_layers)"""
    return np.stack([to_float32(t).ravel() for t in list_of_tensors])

for orig_name, shuf_name in pairs:
    A = stack_batches(data[orig_name])     # shape (100, 35)
    B = stack_batches(data[shuf_name])     # shape (100, 35)
    
    n_batches, n_layers = A.shape

    layer_ix = []
    var_A = []
    var_B = []
    pvals = []

    # Test variance per LAYER (across 100 batches)
    for i in range(n_layers):
        a = A[:, i]
        b = B[:, i]
        vA = np.var(a, ddof=0)
        vB = np.var(b, ddof=0)
        _, p = levene(a, b, center='median')  # Brown–Forsythe
        layer_ix.append(i)
        var_A.append(vA)
        var_B.append(vB)
        pvals.append(p)

    rej, p_adj, _, _ = multipletests(pvals, alpha=ALPHA, method='fdr_bh')

    df = pd.DataFrame({
        "layer": layer_ix,
        "var_original": var_A,
        "var_shuffled": var_B,
        "levene_p": pvals,
        "levene_p_fdr": p_adj,
        "different_variance_FDR": rej
    })

    # Global summary across layers
    u_stat, p_less = mannwhitneyu(var_A, var_B, alternative="less")

    print(f"\n=== {orig_name} vs {shuf_name} ===")
    print(f"{n_layers} layers, {n_batches} batches per layer")
    print(f"Mean variance original: {np.mean(var_A):.6f}")
    print(f"Mean variance shuffled: {np.mean(var_B):.6f}")
    print(f"Mann–Whitney p (orig < shuf): {p_less:.3e}")
    print(f"Layers with FDR<0.05: {np.sum(rej)}/{n_layers}")


=== gpt4_para1 vs gpt4_para1_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000057
Mean variance shuffled: 0.000067
Mann–Whitney p (orig < shuf): 7.830e-02
Layers with FDR<0.05: 6/32

=== gpt4_para2 vs gpt4_para2_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000046
Mean variance shuffled: 0.000056
Mann–Whitney p (orig < shuf): 2.030e-03
Layers with FDR<0.05: 2/32

=== gpt4_para3 vs gpt4_para3_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000029
Mean variance shuffled: 0.000036
Mann–Whitney p (orig < shuf): 5.654e-02
Layers with FDR<0.05: 19/32

=== gpt5_para1 vs gpt5_para1_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000061
Mean variance shuffled: 0.000068
Mann–Whitney p (orig < shuf): 3.560e-01
Layers with FDR<0.05: 0/32

=== gpt5_para2 vs gpt5_para2_shuffled ===
32 layers, 100 batches per layer
Mean variance original: 0.000062
Mean variance shuffled: 0.000062
Mann–Whitney p (orig 